In [19]:
import os
from enum import Enum

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Check the .env file in your project root."
    )

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Imports successful.")
print("Groq model initialized.")

Imports successful.
Groq model initialized.


In [20]:
class Sentiment(str, Enum):
    POSITIVE = "Positive"
    NEGATIVE = "Negative"
    NEUTRAL = "Neutral"


class ComplaintData(BaseModel):
    complaint: str = Field(
        description="The customer's main complaint, based only on the review"
    )

    product_or_feature: str = Field(
        description="The product or feature mentioned in the review"
    )

    sentiment: Sentiment = Field(
        description="Overall customer sentiment: Positive, Negative, or Neutral"
    )


parser = PydanticOutputParser(
    pydantic_object=ComplaintData
)

print("Pydantic schema created.")
print(parser.get_format_instructions())

Pydantic schema created.
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Sentiment": {"enum": ["Positive", "Negative", "Neutral"], "title": "Sentiment", "type": "string"}}, "properties": {"complaint": {"description": "The customer's main complaint, based only on the review", "title": "Complaint", "type": "string"}, "product_or_feature": {"description": "The product or feature mentioned in the review", "title": "Product Or Feature", "type": "string"}, "sentiment": {"$ref": "#/$defs/Sentiment", "description": "Overall customer sentiment: Positive, Negative, or Neutral"}}, "require

In [21]:
class Sentiment(str, Enum):
    POSITIVE = "Positive"
    NEGATIVE = "Negative"
    NEUTRAL = "Neutral"


class ComplaintData(BaseModel):
    complaint: str = Field(
        description="The customer's main complaint, based only on the review"
    )

    product_or_feature: str = Field(
        description="The product or feature mentioned in the review"
    )

    sentiment: Sentiment = Field(
        description="Overall customer sentiment: Positive, Negative, or Neutral"
    )


parser = PydanticOutputParser(
    pydantic_object=ComplaintData
)

print("Pydantic schema created.")
print(parser.get_format_instructions())

Pydantic schema created.
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Sentiment": {"enum": ["Positive", "Negative", "Neutral"], "title": "Sentiment", "type": "string"}}, "properties": {"complaint": {"description": "The customer's main complaint, based only on the review", "title": "Complaint", "type": "string"}, "product_or_feature": {"description": "The product or feature mentioned in the review", "title": "Product Or Feature", "type": "string"}, "sentiment": {"$ref": "#/$defs/Sentiment", "description": "Overall customer sentiment: Positive, Negative, or Neutral"}}, "require

In [22]:
review_text = """
I bought wireless headphones last week. The sound quality is good,
but the battery stops working after only one hour. I am disappointed
because I expected them to last longer.
"""

print("CUSTOMER REVIEW")
print(review_text)

CUSTOMER REVIEW

I bought wireless headphones last week. The sound quality is good,
but the battery stops working after only one hour. I am disappointed
because I expected them to last longer.



In [23]:
extraction_prompt = PromptTemplate(
    template="""
Extract the main complaint, product or feature, and overall sentiment
from the customer review.

Rules:
- Use only information supported by the review.
- Identify the main complaint, even if the review includes positive details.
- Select sentiment based on the overall tone of the review.
- Do not invent information.
- Follow the required output format exactly.

Customer review:
{review_text}

{format_instructions}
""",
    input_variables=["review_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

extraction_chain = extraction_prompt | llm | parser

complaint_data = extraction_chain.invoke({
    "review_text": review_text
})

print("STEP 1 — STRUCTURED COMPLAINT DATA")
print(complaint_data.model_dump_json(indent=2))

STEP 1 — STRUCTURED COMPLAINT DATA
{
  "complaint": "Battery stops working after only one hour (short battery life)",
  "product_or_feature": "wireless headphones",
  "sentiment": "Negative"
}


In [24]:
ticket_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a support ticket writer for an internal customer support team.

Create a concise, useful support ticket using only the supplied
structured complaint data.

Include:
1. Complaint summary
2. Product or feature
3. Customer sentiment
4. Suggested next action for the support team

Rules:
- Do not invent order numbers, account details, warranty status, or dates.
- Do not claim that an action has already been taken.
- Clearly distinguish the reported problem from the suggested next action.
- Return plain text only, without JSON or Markdown code fences."""
    ),
    (
        "human",
        "Structured complaint data:\n{complaint_data_json}"
    )
])

ticket_chain = ticket_prompt | llm | StrOutputParser()

print("Step 2 support ticket chain created.")

Step 2 support ticket chain created.


In [25]:
complaint_data_json = complaint_data.model_dump_json(indent=2)

support_ticket = ticket_chain.invoke({
    "complaint_data_json": complaint_data_json
})

if not isinstance(support_ticket, str) or not support_ticket.strip():
    raise ValueError("The support ticket output is empty.")

print("STEP 2 — SUPPORT TICKET SUMMARY")
print("=" * 45)
print(support_ticket)

STEP 2 — SUPPORT TICKET SUMMARY
Complaint Summary: Battery stops working after only one hour (short battery life)  
Product or Feature: Wireless headphones  
Customer Sentiment: Negative  

Suggested Next Action: Have the support team verify the issue by asking the customer to perform basic battery troubleshooting (e.g., full charge cycle, firmware update). If the problem persists, initiate a warranty claim or arrange a replacement unit.


In [26]:
# Validate Step 1
assert isinstance(complaint_data, ComplaintData)
assert complaint_data.complaint.strip()
assert complaint_data.product_or_feature.strip()
assert isinstance(complaint_data.sentiment, Sentiment)

# Validate Step 2
assert isinstance(support_ticket, str)
assert support_ticket.strip()

# Verify that Step 2's input contains structured data.
assert complaint_data_json == complaint_data.model_dump_json(indent=2)

print("Validation passed.")
print("Step 1 returned valid Pydantic complaint data.")
print("Step 2 returned a non-empty support ticket.")
print("Step 2 uses the structured output from Step 1.")

Validation passed.
Step 1 returned valid Pydantic complaint data.
Step 2 returned a non-empty support ticket.
Step 2 uses the structured output from Step 1.


In [27]:
test_review = """
The new coffee machine looks attractive, but it leaks water
whenever I use it. I am unhappy with this purchase.
"""

test_complaint_data = extraction_chain.invoke({
    "review_text": test_review
})

test_ticket = ticket_chain.invoke({
    "complaint_data_json": test_complaint_data.model_dump_json(indent=2)
})

assert isinstance(test_complaint_data, ComplaintData)
assert test_ticket.strip()

print("SECOND TEST — STRUCTURED DATA")
print(test_complaint_data.model_dump_json(indent=2))

print("\nSECOND TEST — SUPPORT TICKET")
print(test_ticket)

print("\nSecond test passed.")

SECOND TEST — STRUCTURED DATA
{
  "complaint": "It leaks water when used",
  "product_or_feature": "coffee machine",
  "sentiment": "Negative"
}

SECOND TEST — SUPPORT TICKET
Complaint Summary: The coffee machine leaks water during use.  
Product or Feature: Coffee machine  
Customer Sentiment: Negative  
Suggested Next Action: Ask the customer for the model number and purchase details, arrange for a troubleshooting call or return for inspection, and provide a replacement or repair option if the issue is confirmed.

Second test passed.


In [28]:
print("""
Conclusion:
The Customer Review to Support Ticket Pipeline contains two linked
LangChain steps.

Step 1 uses PromptTemplate and PydanticOutputParser to extract the
complaint, product or feature, and customer sentiment into validated
structured data.

Step 2 uses ChatPromptTemplate with a support-ticket-writer persona
and StrOutputParser to generate an internal support ticket using only
the structured data produced by Step 1.

The notebook tests two reviews and validates the output structure.
""")


Conclusion:
The Customer Review to Support Ticket Pipeline contains two linked
LangChain steps.

Step 1 uses PromptTemplate and PydanticOutputParser to extract the
complaint, product or feature, and customer sentiment into validated
structured data.

Step 2 uses ChatPromptTemplate with a support-ticket-writer persona
and StrOutputParser to generate an internal support ticket using only
the structured data produced by Step 1.

The notebook tests two reviews and validates the output structure.

